In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 50
trainset = torchvision.datasets.CIFAR10(root='../data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=0)
testset = torchvision.datasets.CIFAR10(root='../data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=0)

 43%|████▎     | 73.5M/170M [24:41<36:31, 44.3kB/s]  

In [ ]:
Din, H, K = 3 * 32 * 32, 100, 10
std = 1e-2

torch.manual_seed(0)
W1 = torch.randn(Din, H) * std
b1 = torch.zeros(H)
W2 = torch.randn(H, K) * std
b2 = torch.zeros(K)

epochs = 10
lr = 0.05
lr_decay = 0.9

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + torch.exp(-z))

def softmax(z):
    z = z - z.max(dim=1, keepdim=True).values   # subtract max for numerical stability
    e = torch.exp(z)
    return e / e.sum(dim=1, keepdim=True)

def forward(x):
    z1 = x.mm(W1) + b1
    h = sigmoid(z1)
    z2 = h.mm(W2) + b2
    return h, softmax(z2)

In [ ]:
loss_history = []
for t in range(epochs):
    running_loss = 0.0
    for inputs, labels in trainloader:
        N = inputs.shape[0]
        x = inputs.view(N, -1)
        Y = nn.functional.one_hot(labels, K).float()

        # forward
        h, p = forward(x)
        loss = -torch.log(p[torch.arange(N), labels]).mean()
        loss_history.append(loss.item())
        running_loss += loss.item()

        # backward
        dz2 = (p - Y) / N
        dW2 = h.t().mm(dz2)
        db2 = dz2.sum(dim=0)
        dh = dz2.mm(W2.t())
        dz1 = dh * h * (1 - h)
        dW1 = x.t().mm(dz1)
        db1 = dz1.sum(dim=0)

        # update
        W2 -= lr * dW2; b2 -= lr * db2
        W1 -= lr * dW1; b1 -= lr * db1

    print(f"Epoch {t+1}/{epochs}, loss: {running_loss/len(trainloader):.4f}")
    lr *= lr_decay

In [ ]:
def accuracy(loader):
    correct = total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            x = inputs.view(inputs.shape[0], -1)
            _, p = forward(x)
            correct += (p.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

print(f"Training accuracy: {accuracy(trainloader):.2f}%")
print(f"Test accuracy:     {accuracy(testloader):.2f}%")

plt.plot(loss_history)
plt.title("Loss history"); plt.xlabel("Iteration"); plt.ylabel("Loss")
plt.show()